# Notebook 27 — WU20H Sequential Release-Hold Preflight

**CPU only · No real NIfTI staging · No GPU training · No model predictions · No clinical approval.**

This notebook only checks the frozen private JSON/CSV evidence, plans a *one-fold-at-a-time* capacity budget, and exercises an artificial 1,000-event training-access trace. All results remain blocked for actual clinical-data staging and training.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 1. Locate private evidence and verify immutable plans and splits

Neither training image voxels nor reference label files will be opened. A review checklist containing `APPROVED` is **not** an authenticated scientific signature.


In [ ]:
import hashlib, json, os, subprocess, sys, tempfile, urllib.request
from pathlib import Path
ROOT=Path('/content/drive/MyDrive/HybridStrokeSeg/ISLES2024')
RAW=ROOT/'nnUNet_raw'/'Dataset505_ISLES2024_NCCT_PERFUSION'
PRE=ROOT/'nnUNet_preprocessed'/'Dataset505_ISLES2024_NCCT_PERFUSION'
SPLITS=PRE/'splits_final.json'
PLANS=PRE/'nnUNetPlans.json'
FOLDS=ROOT/'results'/'wu20b_full_fold_calibration'/'fold_private'
E=ROOT/'results'/'wu20e_review_and_release_gate'/'WU20E_release_gate_aggregate.json'
POLICY=ROOT/'results'/'wu20e_review_and_release_gate'/'WU20E_policy_PROPOSED_NOT_APPROVED_PRIVATE.json'
REVIEW=ROOT/'results'/'wu20c_nnunet_runtime_qualification'/'WU20C_mask_review_checklist_PRIVATE.csv'
F=ROOT/'results'/'wu20f_virtual_staging_and_forward'
VIRTUAL=F/'WU20F_virtual_staging_PRIVATE.json'
PREFLIGHT=F/'WU20F_preflight_aggregate.json'
G=ROOT/'results'/'wu20g_physical_staging_qualification'/'WU20G_physical_staging_plan_PRIVATE.json'
OUT=ROOT/'results'/'wu20h_sequential_release_hold'
SCRATCH=Path('/content/wu20h_scratch')
SOURCE=Path('/content/wu20h_pinned_source')
def sha256(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as f:
        for part in iter(lambda:f.read(1<<20),b''):h.update(part)
    return h.hexdigest()
assert sha256(SPLITS)=='664ed51e15374f8f9568d586fba2cb87d5157dc90986e5eca82c9c8440bf6027'
assert sha256(PLANS)=='6203521136de2f57bbe3895a639863112a5869697de9b752ade42dc1a23e4dca'
for source in [G,VIRTUAL,PREFLIGHT,E,POLICY,REVIEW]:
    assert source.is_file(), f'Missing private evidence: {source.name}'
assert FOLDS.is_dir() and all((FOLDS/f'WU20B_fold{i}_parameters_PRIVATE.json').is_file() for i in range(5))
SCRATCH.mkdir(parents=True,exist_ok=True)
SOURCE.mkdir(parents=True,exist_ok=True)
OUT.mkdir(parents=True,exist_ok=True)
print('Original immutable hashes verified; source clinical approval is NOT implied')


## 2. Get source from exact reviewed GitHub SHA (not a moving branch)

In [ ]:
COMMIT='fd297607a1a1c9d848eaa4efa048f64d17a0603c'
BASE=f'https://raw.githubusercontent.com/NTinkicht/HybridStrokeSeg-Reproduction/{COMMIT}'
SCRIPTS=('scripts/plan_isles2024_wu20h_sequential.py',
         'scripts/audit_isles2024_wu20h_training_access.py')
for rel in SCRIPTS:
    dst=SOURCE/rel
    dst.parent.mkdir(parents=True,exist_ok=True)
    with urllib.request.urlopen(f'{BASE}/{rel}',timeout=60) as response:
        dst.write_bytes(response.read())
    print(rel,'SHA-256:',sha256(dst))


## 3. Refresh one-fold-at-a-time plan and LIVE scratch free-space budget

This is a metadata-only read and writes only private JSON receipts, not staging files. It never marks training safe.


In [ ]:
CMD=[sys.executable,'-u',str(SOURCE/'scripts'/'plan_isles2024_wu20h_sequential.py'),
     '--mode','plan','--wu20g-plan',str(G),'--wu20f-virtual',str(VIRTUAL),
     '--wu20f-preflight',str(PREFLIGHT),'--wu20e-gate',str(E),'--wu20e-policy',str(POLICY),
     '--review-csv',str(REVIEW),'--splits',str(SPLITS),
     '--plans',str(PLANS),'--folds',str(FOLDS),
     '--scratch',str(SCRATCH),'--output',str(OUT)]
result=subprocess.run(CMD,text=True,capture_output=True,check=False)
if result.returncode:
    print(result.stderr[-6000:]);raise RuntimeError('WU20H sequential planning failed')
REPORT=json.loads((OUT/'WU20H_sequential_aggregate.json').read_text())
assert REPORT['status']=='SEQUENTIAL_CAPACITY_AND_TRAINER_HOLD_PLAN_COMPLETE'
assert REPORT['training_ready'] is False and REPORT['gpu_training_started'] is False
assert REPORT['physical_staging_authorized'] is False
assert REPORT['n_folds']==5 and REPORT['n_patients']==149
print(json.dumps({'plan_status':REPORT['status'],'storage':REPORT['storage'],
                  'clinical_review':REPORT['anatomical_review'],'training_ready':False},indent=2))


## 4. Synthetic 1,000-step access-contract simulation—NO training performed

The generated 1,000 synthetic epoch events represent only a contract unit test, not actual nnU-Net training or effective isolation of clinical labels.


In [ ]:
import shutil
with tempfile.TemporaryDirectory(prefix='wu20h_trace_fixture_') as tmp:
    synthetic=Path(tmp)/'synthetic_trace.json'
    cmd=[sys.executable,'-u',str(SOURCE/'scripts'/'audit_isles2024_wu20h_training_access.py'),
         '--mode','synthetic','--output',str(synthetic)]
    completed=subprocess.run(cmd,text=True,capture_output=True)
    if completed.returncode:
        print(completed.stderr);raise RuntimeError('Synthetic trainer event audit failed')
    result=json.loads(synthetic.read_text())
    assert result['status']=='SYNTHETIC_TRAINING_TRACE_CONTRACT_PASS'
    assert result['synthetic_training_steps']==1000
    assert result['actual_nnunet_trainer_hook_verified'] is False
    assert result['trained_model_executed'] is False and result['training_ready'] is False
    private_target=OUT/'WU20H_training_trace_synthetic_aggregate.json'
    if private_target.exists() and private_target.read_bytes()!=synthetic.read_bytes():
        raise ValueError('Previous trace receipt differs; stop rather than overwrite')
    if not private_target.exists():
        shutil.copy2(synthetic,private_target)
TRACE=json.loads(private_target.read_text())
print(json.dumps(TRACE,indent=2))


## 5. Interpretation and remaining release holds

A passing sequential report **does not** satisfy the authentic independent clinical/scientific approvals, actual training-code instrumentation, full label geometry qualification, full preprocessing/checkpoint storage, or authorization to run patient data on GPUs. No cleanup or physical staging is run here.


In [ ]:
print(json.dumps({'wu20h_status':REPORT['status'],
                  'patient_folds_checked':REPORT['n_folds'],
                  'one_fold_staging_reservation_bytes':REPORT['storage']['conservative_one_fold_2x_reservation_bytes'],
                  'live_scratch_free_bytes':REPORT['storage']['current_scratch_free_bytes'],
                  'staging_capacity_only':REPORT['storage']['provisional_staging_space_meets_2x'],
                  'actual_trainer_hooks_verified':TRACE['actual_nnunet_trainer_hook_verified'],
                  'independent_review_authenticated':False,
                  'physical_staging_authorized':False,'training_ready':False},indent=2))
